# Loading Data


In [2]:
import pandas as pd
import ast
import numpy as np

In [133]:
mbm_bat = pd.read_csv(r'D:\ED\FIFS_IPLungit\data\mbm\cricsheet_t20i_mbm_bat.csv')
playing_xi = pd.read_csv(r'D:\ED\FIFS_IPLungit\data\register_data\playing_xi\t20i_playing_xi.csv')
# player_info = pd.read_csv(path + '/player_details.csv')
player_info = pd.read_csv(r'D:\ED\FIFS_IPLungit\data\register_data\people.csv')

# Cleaning Data

In [136]:
mbm_bat['batter_id'] = mbm_bat['batter_id'].apply(lambda x: str(int(float(x))) if pd.notnull(x) else x)


# Function to parse playing_xi columns
def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Define a helper function that returns the longest non-null string among the three names
def get_longest_name(row):
    names = [row['name'], row['unique_name']]
    # Filter out any missing values
    names = [name for name in names if pd.notnull(name)]
    return max(names, key=len) if names else ''

# Ensure all player IDs in playing_xi are strings
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(lambda x: [str(pid) for pid in x])
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(lambda x: [str(pid) for pid in x])
# and the value is the longest name obtained by applying the function above
player_map = dict(zip(player_info['key_cricinfo'].astype(str),
                      player_info.apply(get_longest_name, axis=1)))

# Handling DNB Values

In [137]:
dnb_rows = []

# Group mbm_bat by match_id and innings (since a match can have multiple innings)
for (match_id, inns), group in mbm_bat.groupby(['match_id', 'inns']):
    # Get common details from the group (assumed to be the same for all rows in the group)
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]
    bowl_team = group['bowl_team'].iloc[0]
    venue = group['venue'].iloc[0]
    league = group['league'].iloc[0]  # Extract league from the group

    # Determine the set of batter IDs (as strings) who batted in this innings
    batted_ids = set(group['batter_id'].unique())

    # Lookup the corresponding playing_xi row for this match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]
    # Determine which playing XI list corresponds to the batting team
    if row['team1_name'] == bat_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bat_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Batting team {bat_team} not found in playing XI teams for match {match_id}.")
        continue

    # Convert all player IDs in the playing XI list to strings
    full_xi = [str(pid) for pid in full_xi]

    # Identify players in the playing XI who did not bat (i.e. DNB)
    dnb_ids = [pid for pid in full_xi if pid not in batted_ids]

    # For each DNB player, create a new row with default values
    for pid in dnb_ids:
        batter_name = player_map.get(pid, "")
        new_row = {
            'date': date,
            'match_id': match_id,
            'bat_team': bat_team,
            'bowl_team': bowl_team,
            'inns': inns,
            'venue': venue,
            'batter_id': pid,
            'batter': batter_name,
            'runs': 0,
            'balls': 0,
            'fours': 0,
            'sixes': 0,
            'sr': 0,
            'isOut': False,
            'wicketType': "DNB",
            'bowler_id': np.nan,
            'bowler': np.nan,
            'league': league  # Include league in the new row
        }
        dnb_rows.append(new_row)

In [138]:
standardized_player_map = {}
for key, value in player_map.items():
    clean_key = str(key).replace('.0', '')
    standardized_player_map[clean_key] = value

# Create the combined dataframe
dnb_df = pd.DataFrame(dnb_rows)
combined_df = pd.concat([mbm_bat, dnb_df], ignore_index=True)

# Update batter names using standardized player map
combined_df['batter'] = combined_df.apply(
    lambda row: row['batter'] if pd.notnull(row['batter']) and row['batter'] != ""
    else standardized_player_map.get(str(row['batter_id']).replace('.0', ''), ""),
    axis=1
)

# Correctly identify DNB rows based on wicketType
combined_df['is_DNB'] = combined_df['wicketType'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB'], inplace=True)

# Safely convert bowler_id to string format
combined_df['bowler_id'] = combined_df['bowler_id'].apply(
    lambda x: str(int(float(x))) if pd.notnull(x) else x
)

In [139]:
combined_df

,match_id,date,venue,inns,batter_id,batter,bowler,bowler_id,runs,balls,fours,sixes,sr,isOut,wicketType,bat_team,bowl_team,league
0,211028,2005-06-13,The Rose Bowl,1,19296,KP Pietersen,MJ Clarke,4578,34,18,3,1,188.89,True,caught,England,Australia,T20I
1,211028,2005-06-13,The Rose Bowl,1,21585,ME Trescothick,A Symonds,7702,41,37,5,0,110.81,True,caught,England,Australia,T20I
2,211028,2005-06-13,The Rose Bowl,1,20387,AJ Strauss,JN Gillespie,5392,18,16,1,0,112.50,True,bowled,England,Australia,T20I
3,211028,2005-06-13,The Rose Bowl,1,22182,MP Vaughan,A Symonds,7702,0,1,0,0,0.00,True,caught,England,Australia,T20I
4,211028,2005-06-13,The Rose Bowl,1,10772,PD Collingwood,GD McGrath,6565,46,26,5,2,176.92,True,caught,England,Australia,T20I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58756,1476798,2025-03-23,"Namibia Cricket Ground, Windhoek",2,1310415,JT Brassell,NaN,NaN,0,0,0,0,0.00,False,DNB,Namibia,Canada,T20I
58757,1476798,2025-03-23,"Namibia Cricket Ground, Windhoek",2,574638,MB Kruger,NaN,NaN,0,0,0,0,0.00,False,DNB,Namibia,Canada,T20I
58758,1476798,2025-03-23,"Namibia Cricket Ground, Windhoek",2,698317,R Trumpelmann,NaN,NaN,0,0,0,0,0.00,False,DNB,Namibia,Canada,T20I
58759,1476798,2025-03-23,"Namibia Cricket Ground, Windhoek",2,524075,T Lungameni,NaN,NaN,0,0,0,0,0.00,False,DNB,Namibia,Canada,T20I


In [140]:
# Convert bowler_id float values to integer strings, preserving NaN values
combined_df['bowler_id'] = combined_df['bowler_id'].apply(lambda x: str(int(float(x))) if pd.notnull(x) else x)

In [141]:
combined_df.to_csv(r"D:\ED\FIFS_IPLungit\data\mbm\final_t20i_mbm_bat.csv", index = False)